In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
from mounirood.framework import FrameworkFactory
from mounirood.
import subprocess

/home/maouche/.conda/envs/multiood/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Test file retrieval

Je pourrais juste comparer les noms mais j'ai pas de fonction qui me les rend directement dans framework et flemme et c'est lisible déjà

In [38]:
from mounirood.tests_integrite import test_all

In [40]:
test_all()

OK


# a

In [21]:
framework = FrameworkFactory("near_ood")
layer_proc = "ash"
tester = TesterVanilla(framework, layer_proc, "UCF")

In [3]:
if framework.ood_mode == "near_ood":
    dataset_id, dataset_ood = framework.get_confs("id", layer_proc)[dataset], framework.get_confs("ood", layer_proc)[dataset]
else:
    dataset_id, dataset_ood = framework.get_confs("id", layer_proc), framework.get_confs("ood", layer_proc)
dataset_id, dataset_ood = dataset_id, dataset_ood

msp = dataset_id[:, 0] 
msp.shape

(877,)

In [18]:
tester.get_auroc_fpr()

scores_id.mean()=0.9932071, scores_id.var()=0.002476296
scores_ood.mean()=0.7155917, scores_ood.var()=0.066934764


(0.042558272971672084, 0.9990291262135922)

In [41]:
command = framework.eval_command_template
params = {"postprocessor":"msp", "backbone":"baseline","sparsification_suffix":"react","modality": '',"dataset": "UCF"}
command = command.format(**params)
command

"python eval_video_flow_near_ood.py --postprocessor msp --appen 'baseline_best_react_' --dataset 'UCF' --path 'HMDB-rgb-flow/' 2>error_eval_UCF_near_ood_react.log | tee out_eval_UCF_near_ood_react.log"

In [40]:
command = framework.eval_command_template
params = {"postprocessor":"msp", "backbone":"baseline","sparsification_suffix":"react","modality": '',"dataset": "UCF"}
command = command.format(**params)
command = command.split()
result = subprocess.run(command, capture_output=True, text=True)
print("stdout:", result.stdout)
print("stderr:", result.stderr)

stdout: 
stderr: usage: eval_video_flow_near_ood.py [-h] [--postprocessor POSTPROCESSOR]
                                   [--appen APPEN] [--dataset DATASET]
                                   [--path PATH] [--resume_file RESUME_FILE]
eval_video_flow_near_ood.py: error: unrecognized arguments: 2>error_eval_UCF_near_ood_react.log | tee out_eval_UCF_near_ood_react.log



In [47]:
%python eval_video_flow_near_ood.py --postprocessor msp --appen 'baseline_best_react_' --dataset 'UCF' --path 'HMDB-rgb-flow/' 2>error_eval_UCF_near_ood_react.log | tee out_eval_UCF_near_ood_react.log

UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


# remake

Erreur

In [30]:
import numpy as np
#from metrics import compute_all_metrics, auc_and_fpr_recall
# from mounirood.eval_functions import auc_and_fpr_recall
from mounirood.datasets import get_y
from sklearn import metrics



def auc_and_fpr_recall(conf, label, tpr_th):
    # following convention in ML we treat OOD as positive
    #ood_indicator = np.zeros_like(label)
    #ood_indicator[label == -1] = 1
    ood_indicator = label
    
    
    # in the postprocessor we assume ID samples will have larger
    # "conf" values than OOD samples
    # therefore here we need to negate the "conf" values
    
    # Ici, j'ai inversé les signes des 3 confs suivants
    # Parce que dans le classifieur, comme la classe OOD est 1, plus le "score" / logit
    # d'un exemple est élevé, plus il sera classifié comme OOD. Il faut donc inverser le paradigme.
    fpr_list, tpr_list, thresholds = metrics.roc_curve(ood_indicator, conf)
    fpr = fpr_list[np.argmax(tpr_list >= tpr_th)]

    precision_in, recall_in, thresholds_in \
        = metrics.precision_recall_curve(1 - ood_indicator, -conf)

    precision_out, recall_out, thresholds_out \
        = metrics.precision_recall_curve(ood_indicator, conf)

    auroc = metrics.auc(fpr_list, tpr_list)
    aupr_in = metrics.auc(recall_in, precision_in)
    aupr_out = metrics.auc(recall_out, precision_out)

    return auroc, aupr_in, aupr_out, fpr



def acc(pred, label):
    ind_pred = pred[label != -1]
    ind_label = label[label != -1]

    num_tp = np.sum(ind_pred == ind_label)
    acc = num_tp / len(ind_label)

    return acc

normalizer = lambda x: x / np.linalg.norm(x, axis=-1, keepdims=True) + 1e-10

# parser = argparse.ArgumentParser()
# parser.add_argument("--postprocessor", type=str, default='msp') # 'msp' 'ebo' 'maxlogit' 'Mahalanobis' 'ash' 'react' 'knn' 'gen' 'vim'
# parser.add_argument("--appen", type=str, default='a2d_npmix_best_') # a2d_npmix_best_ a2d_npmix_best_ash_ a2d_npmix_best_react_
# parser.add_argument("--dataset", type=str, default='Kinetics') # HMDB UCF Kinetics EPIC
# parser.add_argument("--path", type=str, default='HMDB-rgb-flow') # HMDB-rgb-flow EPIC-rgb-flow
# parser.add_argument("--resume_file", type=str, default='HMDB-rgb-flow/models/checkpoint.pt') # for vim 'HMDB_near_ood_a2d_npmix.pt'
# args = parser.parse_args()

args = {"postprocessor": "msp", "appen": 'baseline_best_', "dataset": 'UCF', "path": 'HMDB-rgb-flow/'}
#react_
if args["dataset"] == 'HMDB':
    num_classes = 25
elif args["dataset"] == 'UCF':
    num_classes = 50
elif args["dataset"] == 'Kinetics':
    num_classes = 129
elif args["dataset"] == 'EPIC':
    num_classes = 4

split = 'test'
print(split)

# output_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_output_' + args["appen"] + split + '.npy'
# pred_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_pred_' + args["appen"] + split + '.npy'
# conf_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_conf_' + args["appen"] + split + '.npy'
label_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_label_' + args["appen"] + split + '.npy'
# feature_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_feature_' + args["appen"] + split + '.npy'
# id_output = np.load(output_name)
# id_pred = np.load(pred_name)
# id_conf = np.load(conf_name)
id_gt = np.load(label_name)
# id_feature = np.load(feature_name)


# ID_ACC = acc(id_pred, id_gt)
# print("ID_ACC: ", ID_ACC)

split = 'eval'
print(split)

# output_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_output_' + args["appen"] + split + '.npy'
# pred_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_pred_' + args["appen"] + split + '.npy'
# conf_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_conf_' + args["appen"] + split + '.npy'
#label_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_label_' + args["appen"] + split + '.npy'
# feature_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_feature_' + args["appen"] + split + '.npy'
# ood_output = np.load(output_name)
# ood_pred = np.load(pred_name)
# ood_conf = np.load(conf_name)
#ood_gt = np.load(label_name)
# ood_feature = np.load(feature_name)

framework = FrameworkFactory("near_ood", args["dataset"])
dataset_id = framework.get_confs("id", "ash")[args["dataset"]]
id_conf = dataset_id[:,0]
dataset_ood = framework.get_confs("ood", "ash")[args["dataset"]]
ood_conf = dataset_ood[:,0]

conf = np.vstack((np.reshape(id_conf, (-1,1)), np.reshape(ood_conf, (-1,1))))
label = get_y(dataset_id,dataset_ood).astype(int) 
#
# ood_gt = -1 * np.ones_like(ood_gt)  # hard set to -1 as ood
# ##pred = np.concatenate([id_pred, ood_pred])
# conf = np.concatenate([id_conf, ood_conf])
# label = np.concatenate([id_gt, ood_gt])
#ood_metrics = compute_all_metrics(conf, label, pred)
auroc, aupr_in, aupr_out, fpr = auc_and_fpr_recall(conf, label, 0.95)
print("FPR@95: ", fpr)
print("AUROC: ", auroc)
conf_err, label_err = conf, label

test
eval
FPR@95:  0.9990291262135922
AUROC:  0.042558272971672084


Correct

In [32]:
import numpy as np
#from metrics import compute_all_metrics, auc_and_fpr_recall
from mounirood.datasets import get_y
from sklearn import metrics

def acc(pred, label):
    ind_pred = pred[label != -1]
    ind_label = label[label != -1]

    num_tp = np.sum(ind_pred == ind_label)
    acc = num_tp / len(ind_label)

    return acc


def auc_and_fpr_recall(conf, label, tpr_th):
    # following convention in ML we treat OOD as positive
    ood_indicator = np.zeros_like(label)
    ood_indicator[label == -1] = 1
    
    # in the postprocessor we assume ID samples will have larger
    # "conf" values than OOD samples
    # therefore here we need to negate the "conf" values
    
    # Ici, j'ai inversé les signes des 3 confs suivants
    # Parce que dans le classifieur, comme la classe OOD est 1, plus le "score" / logit
    # d'un exemple est élevé, plus il sera classifié comme OOD. Il faut donc inverser le paradigme.
    fpr_list, tpr_list, thresholds = metrics.roc_curve(ood_indicator, conf)
    fpr = fpr_list[np.argmax(tpr_list >= tpr_th)]

    precision_in, recall_in, thresholds_in \
        = metrics.precision_recall_curve(1 - ood_indicator, -conf)

    precision_out, recall_out, thresholds_out \
        = metrics.precision_recall_curve(ood_indicator, conf)

    auroc = metrics.auc(fpr_list, tpr_list)
    aupr_in = metrics.auc(recall_in, precision_in)
    aupr_out = metrics.auc(recall_out, precision_out)

    return auroc, aupr_in, aupr_out, fpr

def auc_and_fpr_recall(conf, label, tpr_th):
    # following convention in ML we treat OOD as positive
    ood_indicator = np.zeros_like(label)
    ood_indicator[label == -1] = 1

    # in the postprocessor we assume ID samples will have larger
    # "conf" values than OOD samples
    # therefore here we need to negate the "conf" values
    fpr_list, tpr_list, thresholds = metrics.roc_curve(ood_indicator, -conf)
    fpr = fpr_list[np.argmax(tpr_list >= tpr_th)]

    precision_in, recall_in, thresholds_in \
        = metrics.precision_recall_curve(1 - ood_indicator, conf)

    precision_out, recall_out, thresholds_out \
        = metrics.precision_recall_curve(ood_indicator, -conf)

    auroc = metrics.auc(fpr_list, tpr_list)
    aupr_in = metrics.auc(recall_in, precision_in)
    aupr_out = metrics.auc(recall_out, precision_out)

    return auroc, aupr_in, aupr_out, fpr



normalizer = lambda x: x / np.linalg.norm(x, axis=-1, keepdims=True) + 1e-10

# parser = argparse.ArgumentParser()
# parser.add_argument("--postprocessor", type=str, default='msp') # 'msp' 'ebo' 'maxlogit' 'Mahalanobis' 'ash' 'react' 'knn' 'gen' 'vim'
# parser.add_argument("--appen", type=str, default='a2d_npmix_best_') # a2d_npmix_best_ a2d_npmix_best_ash_ a2d_npmix_best_react_
# parser.add_argument("--dataset", type=str, default='Kinetics') # HMDB UCF Kinetics EPIC
# parser.add_argument("--path", type=str, default='HMDB-rgb-flow') # HMDB-rgb-flow EPIC-rgb-flow
# parser.add_argument("--resume_file", type=str, default='HMDB-rgb-flow/models/checkpoint.pt') # for vim 'HMDB_near_ood_a2d_npmix.pt'
# args = parser.parse_args()

args = {"postprocessor": "msp", "appen": 'baseline_best_', "dataset": 'UCF', "path": 'HMDB-rgb-flow/'}
#react_

if args["dataset"] == 'HMDB':
    num_classes = 25
elif args["dataset"] == 'UCF':
    num_classes = 50
elif args["dataset"] == 'Kinetics':
    num_classes = 129
elif args["dataset"] == 'EPIC':
    num_classes = 4

split = 'test'
print(split)

# output_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_output_' + args["appen"] + split + '.npy'
# pred_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_pred_' + args["appen"] + split + '.npy'
# conf_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_conf_' + args["appen"] + split + '.npy'
label_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_label_' + args["appen"] + split + '.npy'
# feature_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_feature_' + args["appen"] + split + '.npy'
# id_output = np.load(output_name)
# id_pred = np.load(pred_name)
# id_conf = np.load(conf_name)
id_gt = np.load(label_name)
# id_feature = np.load(feature_name)


# ID_ACC = acc(id_pred, id_gt)
# print("ID_ACC: ", ID_ACC)

split = 'eval'
print(split)

# output_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_output_' + args["appen"] + split + '.npy'
# pred_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_pred_' + args["appen"] + split + '.npy'
# conf_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_conf_' + args["appen"] + split + '.npy'
label_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_label_' + args["appen"] + split + '.npy'
# feature_name = args["path"] + '/saved_files/id_'+args["dataset"]+'_near_ood_feature_' + args["appen"] + split + '.npy'
# ood_output = np.load(output_name)
# ood_pred = np.load(pred_name)
# ood_conf = np.load(conf_name)
ood_gt = np.load(label_name)
# ood_feature = np.load(feature_name)

framework = FrameworkFactory("near_ood", args["dataset"])
dataset_id = framework.get_confs("id", "ash")[args["dataset"]]
id_conf = dataset_id[:,0]
dataset_ood = framework.get_confs("ood", "ash")[args["dataset"]]
ood_conf = dataset_ood[:,0]

#conf = np.vstack((np.reshape(id_conf, (-1,1)), np.reshape(ood_conf, (-1,1)))) #err
#label = get_y(dataset_id,dataset_ood)  # err

ood_gt = -1 * np.ones_like(ood_gt)  # hard set to -1 as ood
##pred = np.concatenate([id_pred, ood_pred])
conf = np.concatenate([id_conf, ood_conf])
label = np.concatenate([id_gt, ood_gt])
#ood_metrics = compute_all_metrics(conf, label, pred)

auroc, aupr_in, aupr_out, fpr = auc_and_fpr_recall(conf, label, 0.95)
print("FPR@95: ", fpr)
print("AUROC: ", auroc)

test
eval
FPR@95:  0.2203883495145631
AUROC:  0.9574417270283279


In [4]:
np.all(conf.reshape(-1,1) == conf_err)

True

In [20]:
# Comptage code papier
ood_indicator = np.zeros_like(label)
ood_indicator[label == -1] = 1
unique, counts = np.unique(ood_indicator, return_counts=True)
print(unique, counts)

# Comptage mon code
unique, counts = np.unique(label_err, return_counts=True)
print(unique, counts)
np.all(ood_indicator.reshape(-1,1) == label_err)

[0 1] [1030 6616]
[0 1] [1030 6616]


True

# autre